# PySpark Fundamentals

**PySpark** is the Python API for Apache Spark. It provides the same capabilities as Spark's Scala API but through Python syntax. On Databricks, PySpark is available out-of-the-box on every cluster — no installation needed.

## SparkSession — The Entry Point

Every PySpark application starts with a `SparkSession`. On Databricks, it is pre-created as the variable `spark`.

In [ ]:
# On Databricks: SparkSession is pre-available as `spark`
# On a local environment or standalone cluster:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MyApp")           # Application name visible in Spark UI
    .config("spark.some.config", "value")  # Optional configuration
    .getOrCreate()              # Reuses existing session if one exists
)

# Check Spark version
print(spark.version)

## Creating DataFrames

Three main ways to create a DataFrame in PySpark:

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# ── Method 1: From a Python list ──────────────────────────────────────────────
data = [
    (1, "Alice", "Engineering", 95000.0),
    (2, "Bob",   "Marketing",   72000.0),
    (3, "Carol", "Engineering", 105000.0),
    (4, "Dave",  "Marketing",   68000.0),
    (5, "Eve",   "Engineering", 89000.0),
]

# With explicit schema (recommended — avoids type inference)
schema = StructType([
    StructField("id",         IntegerType(), nullable=False),
    StructField("name",       StringType(),  nullable=True),
    StructField("department", StringType(),  nullable=True),
    StructField("salary",     DoubleType(),  nullable=True),
])

df = spark.createDataFrame(data, schema=schema)
df.show()
df.printSchema()

In [ ]:
# ── Method 2: Reading from a file ─────────────────────────────────────────────
# Databricks paths use DBFS (Databricks File System) or cloud storage URIs

# CSV
df_csv = (
    spark.read
    .option("header", "true")       # First row as column names
    .option("inferSchema", "true")  # Infer column types (avoid in production)
    .csv("/path/to/file.csv")
)

# Parquet (preferred format for analytics)
df_parquet = spark.read.parquet("/path/to/data.parquet")

# JSON
df_json = spark.read.json("/path/to/data.json")

# Delta table (Databricks)
df_delta = spark.read.format("delta").load("/path/to/delta_table")
# Or using the catalog:
df_table = spark.table("my_catalog.my_schema.my_table")

## DataFrame Inspection

In [ ]:
# ── Exploration ───────────────────────────────────────────────────────────────
df.show()                  # Display first 20 rows (truncated)
df.show(5, truncate=False) # Show 5 rows, no truncation
df.printSchema()           # Show column names, types, and nullability
df.dtypes                  # List of (column_name, dtype) tuples
df.columns                 # List of column names
df.count()                 # Number of rows  ← ACTION (triggers execution)
df.describe().show()       # Summary statistics (count, mean, stddev, min, max)

# Inspect the physical and logical plans
df.explain()               # Physical plan
df.explain(True)           # Parsed, Analyzed, Optimized + Physical plans

## Core Transformations

### Selecting and Filtering

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when

# ── select() ──────────────────────────────────────────────────────────────────
df.select("name", "salary").show()
df.select(col("name"), col("salary") * 1.1).show()  # 10% raise

# ── filter() / where() ────────────────────────────────────────────────────────
df.filter(col("salary") > 80000).show()
df.where(col("department") == "Engineering").show()

# Multiple conditions: use & (AND), | (OR), ~ (NOT)
df.filter((col("department") == "Engineering") & (col("salary") > 90000)).show()

# ── withColumn() — add or modify a column ─────────────────────────────────────
df = df.withColumn("salary_usd_k", col("salary") / 1000)    # new derived column
df = df.withColumn("salary", col("salary") * 1.05)           # modify existing

# ── Conditional logic with when/otherwise ────────────────────────────────────
df = df.withColumn(
    "level",
    when(col("salary") >= 100000, "Senior")
    .when(col("salary") >= 80000,  "Mid")
    .otherwise("Junior")
)

# ── Renaming and dropping ─────────────────────────────────────────────────────
df.withColumnRenamed("salary", "annual_salary")
df.drop("salary_usd_k")

### Aggregations and GroupBy

In [ ]:
# ── groupBy().agg() ───────────────────────────────────────────────────────────
(
    df.groupBy("department")
    .agg(
        F.count("id").alias("headcount"),
        F.avg("salary").alias("avg_salary"),
        F.max("salary").alias("max_salary"),
        F.min("salary").alias("min_salary"),
        F.sum("salary").alias("total_payroll"),
    )
    .orderBy(F.desc("avg_salary"))
    .show()
)

# ── Common aggregate functions ────────────────────────────────────────────────
# F.count(), F.countDistinct(), F.sum(), F.avg(), F.mean()
# F.max(), F.min(), F.stddev(), F.variance()
# F.collect_list(), F.collect_set()  ← return arrays
# F.first(), F.last()

# ── Pivot table ───────────────────────────────────────────────────────────────
# df.groupBy("year").pivot("department").agg(F.avg("salary")).show()

### Joins

In [ ]:
# Sample second DataFrame
dept_data = [("Engineering", "Alice M."), ("Marketing", "Bob K.")]
dept_df = spark.createDataFrame(dept_data, ["department", "manager"])

# ── Join types ─────────────────────────────────────────────────────────────────
# inner, left (left_outer), right (right_outer), full (outer), semi, anti, cross

# Inner join
df.join(dept_df, on="department", how="inner").show()

# Left join — keeps all rows from left, NULLs for no match on right
df.join(dept_df, on="department", how="left").show()

# Join on multiple columns
df.join(dept_df, on=["department"], how="inner").show()

# Anti join — rows in left that do NOT exist in right
df.join(dept_df, on="department", how="anti").show()

# ── Broadcast join (optimization for small tables) ────────────────────────────
# Forces the small table to be sent to every executor node, avoiding a shuffle
from pyspark.sql.functions import broadcast
df.join(broadcast(dept_df), on="department", how="inner").show()

# Threshold: spark.sql.autoBroadcastJoinThreshold = 10MB by default
# Databricks AQE can automatically switch to broadcast joins at runtime

### Window Functions

In [ ]:
from pyspark.sql.window import Window

# ── Define a window spec ──────────────────────────────────────────────────────
window_dept = (
    Window
    .partitionBy("department")  # Group by department
    .orderBy(F.desc("salary"))  # Order within group
)

# ── Ranking functions ─────────────────────────────────────────────────────────
df = df.withColumn("rank",        F.rank()      .over(window_dept))  # gaps for ties
df = df.withColumn("dense_rank",  F.dense_rank().over(window_dept))  # no gaps
df = df.withColumn("row_number",  F.row_number().over(window_dept))  # unique

# ── Analytic functions ────────────────────────────────────────────────────────
df = df.withColumn("dept_avg_salary", F.avg("salary").over(
    Window.partitionBy("department")
))

# ── Running total (cumulative sum) ────────────────────────────────────────────
df = df.withColumn("running_total", F.sum("salary").over(
    Window.partitionBy("department").orderBy("id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
))

# ── Lag / Lead ────────────────────────────────────────────────────────────────
df = df.withColumn("prev_salary", F.lag("salary", 1).over(window_dept))
df = df.withColumn("next_salary", F.lead("salary", 1).over(window_dept))

df.select("name", "department", "salary", "rank", "dept_avg_salary").show()

## Working with Null Values

In [ ]:
# ── Checking for nulls ────────────────────────────────────────────────────────
df.filter(col("salary").isNull()).show()    # rows WHERE salary IS NULL
df.filter(col("salary").isNotNull()).show() # rows WHERE salary IS NOT NULL

# ── Null count per column ─────────────────────────────────────────────────────
df.select([F.count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# ── Dropping rows with nulls ──────────────────────────────────────────────────
df.dropna()                         # drop rows with ANY null
df.dropna(subset=["salary"])        # drop rows where salary is null
df.dropna(how="all")                # drop rows where ALL columns are null

# ── Filling nulls ────────────────────────────────────────────────────────────
df.fillna(0)                        # fill all numeric nulls with 0
df.fillna({"salary": 50000.0, "department": "Unknown"})  # column-specific

## String Functions

In [ ]:
# ── Common string functions ───────────────────────────────────────────────────
df.withColumn("name_upper",  F.upper(col("name")))
df.withColumn("name_lower",  F.lower(col("name")))
df.withColumn("name_len",    F.length(col("name")))
df.withColumn("name_trim",   F.trim(col("name")))
df.withColumn("name_starts", F.substring(col("name"), 1, 3))  # first 3 chars
df.withColumn("name_split",  F.split(col("name"), " "))       # split to array
df.withColumn("name_concat", F.concat(col("name"), lit(" - "), col("department")))
df.withColumn("dept_replace", F.regexp_replace(col("department"), "Eng", "Eng."))

# ── Filter using pattern matching ─────────────────────────────────────────────
df.filter(col("name").like("A%"))                     # SQL LIKE
df.filter(col("name").rlike("^[A-C]"))                # Regex
df.filter(col("department").isin("Engineering", "Marketing"))  # IN list

## Date and Timestamp Functions

In [ ]:
# ── Common date functions ─────────────────────────────────────────────────────
df = df.withColumn("today", F.current_date())
df = df.withColumn("now",   F.current_timestamp())

# Parsing strings to date/timestamp
df = df.withColumn("hire_date", F.to_date(lit("2023-06-15"), "yyyy-MM-dd"))

# Extracting parts
df.withColumn("year",    F.year(col("hire_date")))
df.withColumn("month",   F.month(col("hire_date")))
df.withColumn("day",     F.dayofmonth(col("hire_date")))
df.withColumn("quarter", F.quarter(col("hire_date")))

# Date arithmetic
df.withColumn("30_days_later", F.date_add(col("hire_date"), 30))
df.withColumn("days_employed", F.datediff(F.current_date(), col("hire_date")))
df.withColumn("months_ago",    F.months_between(F.current_date(), col("hire_date")))

## Writing Data

In [ ]:
# ── Write modes ───────────────────────────────────────────────────────────────
# overwrite  → replace the entire target
# append     → add rows to existing target
# ignore     → do nothing if target already exists
# error      → raise an error if target exists (default)

# ── Write as Parquet ──────────────────────────────────────────────────────────
df.write.mode("overwrite").parquet("/path/to/output/")

# ── Write as Delta (Databricks recommended format) ────────────────────────────
df.write.mode("overwrite").format("delta").save("/path/to/delta_output/")

# ── Write to a Delta table in the catalog ────────────────────────────────────
df.write.mode("overwrite").saveAsTable("my_catalog.my_schema.employees")

# ── Write as CSV ─────────────────────────────────────────────────────────────
df.write.mode("overwrite").option("header", "true").csv("/path/to/csv_output/")

# ── Partitioned write (store files partitioned by column) ─────────────────────
df.write.mode("overwrite").partitionBy("department").format("delta").save("/path/")
# Creates subdirectories: /path/department=Engineering/ and /path/department=Marketing/

## Spark SQL

PySpark allows mixing Python DataFrame API with SQL. Both are equally performant — both use Catalyst.

In [ ]:
# ── Register a DataFrame as a temporary view ──────────────────────────────────
df.createOrReplaceTempView("employees")        # session-scoped
df.createOrReplaceGlobalTempView("employees")  # app-scoped (use global_temp.employees)

# ── Run Spark SQL ─────────────────────────────────────────────────────────────
result = spark.sql("""
    SELECT
        department,
        COUNT(*) AS headcount,
        AVG(salary) AS avg_salary,
        MAX(salary) AS max_salary
    FROM employees
    WHERE salary > 70000
    GROUP BY department
    ORDER BY avg_salary DESC
""")
result.show()

# ── On Databricks: use %sql magic in a notebook cell ─────────────────────────
# %sql
# SELECT * FROM my_catalog.my_schema.employees LIMIT 10;

## User Defined Functions (UDFs)

> ⚠️ **Use built-in functions first!** UDFs break Catalyst optimization and have serialization overhead. Only use when no built-in equivalent exists.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# ── Python UDF (slowest — row-by-row serialization between JVM and Python) ────
def grade_salary(salary):
    if salary is None:
        return "Unknown"
    elif salary >= 100000:
        return "High"
    elif salary >= 75000:
        return "Medium"
    else:
        return "Low"

grade_salary_udf = udf(grade_salary, StringType())

# Register and use
df.withColumn("salary_grade", grade_salary_udf(col("salary"))).show()

# ── Pandas UDF / Vectorized UDF (much faster — uses Apache Arrow) ─────────────
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(StringType())
def grade_salary_pandas(salary_series: pd.Series) -> pd.Series:
    return salary_series.apply(lambda s: "High" if s >= 100000 else "Medium" if s >= 75000 else "Low")

df.withColumn("salary_grade", grade_salary_pandas(col("salary"))).show()

## Performance Tips Summary

| Tip | Why |
|-----|-----|
| ✅ Use built-in `pyspark.sql.functions` over UDFs | Optimized by Catalyst; runs in JVM |
| ✅ Use Parquet or Delta format | Columnar storage, compression, predicate pushdown |
| ✅ Push filters early (`.filter()` before `.join()`) | Reduces data volume before shuffle |
| ✅ Use `broadcast()` for small tables in joins | Eliminates shuffle |
| ✅ Partition data on high-cardinality filter columns | Enables partition pruning |
| ✅ Cache DataFrames used multiple times | Avoids recomputation |
| ✅ Use `coalesce()` before writing (not `repartition`) | Avoids shuffle when reducing partitions |
| ✅ Enable AQE (Adaptive Query Execution) | Auto-optimizes at runtime (default in Databricks) |
| ❌ Avoid `collect()` on large DataFrames | Sends all data to driver — OutOfMemoryError risk |
| ❌ Avoid `for` loops over DataFrame rows | Not distributed; defeats the purpose of Spark |
| ❌ Avoid wide UDFs unless vectorized | Serialization overhead |